# 🚀 Quick Start: Tabular Foundation Models in 5 Minutes

This notebook gets you from zero to predictions with **every major TFM** in under 5 minutes.  
No preprocessing. No hyperparameter tuning. Just raw data in, predictions out.

---

## 1. Setup & Data Loading

In [ ]:
# Install dependencies (uncomment as needed)
# !pip install tabpfn tabicl xgboost catboost lightgbm scikit-learn pandas matplotlib

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss, classification_report
import time

In [ ]:
# Load the "Give Me Some Credit" dataset
# Option A: From Kaggle (if you have it downloaded)
# df = pd.read_csv('data/give_me_credit/cs-training.csv', index_col=0)

# Option B: Use sklearn's built-in credit-like dataset as demo
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=5000, n_features=12, n_informative=8,
    n_redundant=2, weights=[0.93, 0.07],  # ~7% positive (credit default)
    random_state=42, flip_y=0.03
)

# Add some missing values (realistic for credit data)
rng = np.random.RandomState(42)
mask = rng.random(X.shape) < 0.05
X = X.astype(float)
X[mask] = np.nan

feature_names = ['RevolvingUtil', 'Age', 'PastDue30', 'DebtRatio', 'Income',
                 'OpenCredits', 'PastDue90', 'RealEstateLoans', 'PastDue60',
                 'Dependents', 'Feature11', 'Feature12']

X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name='default')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Positive rate: {y_train.mean():.3f}")
print(f"Missing values: {X_train.isna().sum().sum()} cells")

## 2. Results Tracker

In [ ]:
results = []

def evaluate_model(name, y_true, y_proba, elapsed):
    """Compute metrics and store results."""
    auc = roc_auc_score(y_true, y_proba)
    ll = log_loss(y_true, y_proba)
    results.append({'Model': name, 'AUC-ROC': auc, 'Log-Loss': ll, 'Time (s)': elapsed})
    print(f"  ✅ {name:<25} AUC={auc:.4f}  LogLoss={ll:.4f}  Time={elapsed:.2f}s")

---
## 3. TabPFN v2 — The Nature 2025 Model

**License:** Commercial OK with attribution | **Max:** 10K rows, 500 features  
**Key idea:** Passes your training data as "context" and predicts in one forward pass.

In [ ]:
try:
    from tabpfn import TabPFNClassifier

    t0 = time.time()
    clf = TabPFNClassifier(device='cpu')  # Change to 'cuda' if GPU available
    clf.fit(X_train, y_train)             # Stores data as context (no gradient updates!)
    proba = clf.predict_proba(X_test)[:, 1]  # Single forward pass
    elapsed = time.time() - t0

    evaluate_model('TabPFN v2', y_test, proba, elapsed)

except ImportError:
    print('  ⚠️  TabPFN not installed. Run: pip install tabpfn')

## 4. TabPFN v2.5 — Latest & Most Powerful

**License:** ⚠️ NON-COMMERCIAL (research/eval only) | **Max:** 50K rows, 2K features  
**Requires:** HuggingFace login + license acceptance

In [ ]:
try:
    from tabpfn import TabPFNClassifier

    t0 = time.time()
    # Default v2.5 classifier = Real-TabPFN-2.5 (fine-tuned on real data)
    clf_25 = TabPFNClassifier(device='cpu', n_estimators=4)
    clf_25.fit(X_train, y_train)
    proba_25 = clf_25.predict_proba(X_test)[:, 1]
    elapsed = time.time() - t0

    evaluate_model('TabPFN v2.5 (Real)', y_test, proba_25, elapsed)

except Exception as e:
    print(f'  ⚠️  TabPFN v2.5 not available: {e}')
    print('     You need: huggingface-cli login + accept license at')
    print('     https://huggingface.co/Prior-Labs/tabpfn_2_5')

## 5. TabICLv2 — Best Open-Source TFM

**License:** ✅ BSD-3-Clause (fully commercial) | **Max:** 1M+ rows, 2K features  
**Key advantage:** Fastest TFM, 10x faster than TabPFN-2.5, fully open.

In [ ]:
try:
    from tabicl import TabICLClassifier

    t0 = time.time()
    clf_icl = TabICLClassifier(n_estimators=8, version='v2')
    clf_icl.fit(X_train, y_train)
    proba_icl = clf_icl.predict_proba(X_test)[:, 1]
    elapsed = time.time() - t0

    evaluate_model('TabICLv2', y_test, proba_icl, elapsed)

except ImportError:
    print('  ⚠️  TabICL not installed. Run: pip install tabicl')

## 6. Fine-Tuning TabPFN (v2)

When zero-shot isn't enough, TabPFN supports gradient-based fine-tuning.  
This adapts the pre-trained weights to your specific domain.

In [ ]:
try:
    from tabpfn.finetuning import FinetunedTabPFNClassifier

    t0 = time.time()
    ft_clf = FinetunedTabPFNClassifier(
        device='cpu',       # 'cuda' recommended for speed
        n_epochs=10,        # Use 30 for production; 10 for demo speed
        learning_rate=1e-5, # Recommended default
        batch_size=20,
    )
    ft_clf.fit(X_train, y_train)
    proba_ft = ft_clf.predict_proba(X_test)[:, 1]
    elapsed = time.time() - t0

    evaluate_model('TabPFN v2 (Fine-Tuned)', y_test, proba_ft, elapsed)

except Exception as e:
    print(f'  ⚠️  Fine-tuning not available: {e}')

## 7. GBDT Baselines

The "gold standard" to beat. All fully open-source and commercially usable.

In [ ]:
from sklearn.impute import SimpleImputer

# Impute for GBDTs (some don't handle NaN natively)
imp = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imp.transform(X_test), columns=X_test.columns)

# --- XGBoost ---
try:
    from xgboost import XGBClassifier
    t0 = time.time()
    xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='auc',
                         tree_method='hist', enable_categorical=True, verbosity=0)
    xgb.fit(X_train, y_train)  # XGBoost handles NaN natively
    proba_xgb = xgb.predict_proba(X_test)[:, 1]
    evaluate_model('XGBoost (default)', y_test, proba_xgb, time.time() - t0)
except ImportError:
    print('  ⚠️  XGBoost not installed')

# --- CatBoost ---
try:
    from catboost import CatBoostClassifier
    t0 = time.time()
    cb = CatBoostClassifier(iterations=100, random_seed=42, verbose=0)
    cb.fit(X_train_imp, y_train)
    proba_cb = cb.predict_proba(X_test_imp)[:, 1]
    evaluate_model('CatBoost (default)', y_test, proba_cb, time.time() - t0)
except ImportError:
    print('  ⚠️  CatBoost not installed')

# --- LightGBM ---
try:
    from lightgbm import LGBMClassifier
    t0 = time.time()
    lgb = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
    lgb.fit(X_train_imp, y_train)
    proba_lgb = lgb.predict_proba(X_test_imp)[:, 1]
    evaluate_model('LightGBM (default)', y_test, proba_lgb, time.time() - t0)
except ImportError:
    print('  ⚠️  LightGBM not installed')

# --- Random Forest ---
from sklearn.ensemble import RandomForestClassifier
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_imp, y_train)
proba_rf = rf.predict_proba(X_test_imp)[:, 1]
evaluate_model('RandomForest', y_test, proba_rf, time.time() - t0)

---
## 8. 🏆 Final Leaderboard

In [ ]:
if results:
    leaderboard = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
    leaderboard.index = range(1, len(leaderboard) + 1)
    leaderboard.index.name = 'Rank'
    print('\n' + '='*70)
    print('  LEADERBOARD')
    print('='*70)
    display(leaderboard.style.format({'AUC-ROC': '{:.4f}', 'Log-Loss': '{:.4f}', 'Time (s)': '{:.2f}'}))
else:
    print('No results collected. Install at least one model package.')

In [ ]:
# Quick bar chart
if results:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    lb = pd.DataFrame(results).sort_values('AUC-ROC')
    colors = ['#4CAF50' if 'Tab' in m or 'Mitra' in m else '#F44336' for m in lb['Model']]
    ax.barh(lb['Model'], lb['AUC-ROC'], color=colors, edgecolor='white')
    ax.set_xlabel('AUC-ROC')
    ax.set_title('TFMs (green) vs Traditional Models (red)', fontweight='bold')
    for i, (_, row) in enumerate(lb.iterrows()):
        ax.text(row['AUC-ROC'] + 0.002, i, f"{row['AUC-ROC']:.4f}", va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 📝 Key Takeaways

| What | Finding |
|------|--------|
| **Best zero-shot accuracy** | TabPFN v2.5 or TabICLv2 typically win on datasets < 50K rows |
| **Best open-source option** | TabICLv2 (BSD-3, scales to 1M rows, 10x faster than TabPFN-2.5) |
| **Best for enterprise** | TabICLv2 or Mitra (Apache 2.0) — no license restrictions |
| **Fine-tuning helps?** | Usually +0.5-2% AUC improvement, worth it for production |
| **TFMs vs GBDTs** | TFMs match or beat default GBDTs; tuned GBDTs are competitive |
| **Ensemble advice** | TFM + GBDT ensemble almost always beats either alone |

### Next Steps
- Run `02_zero_shot_benchmark.ipynb` for the full evaluation
- Run `04_scaling_experiments.ipynb` to see where models break
- Read the main README.md for the complete experimental design